Import the data for the required area and dates

In [26]:
import requests

""" API docs: https://data.police.uk/docs/ """

""" Define the API endpoint """
outcomes_api_url = "https://data.police.uk/api/outcomes-at-location"
stops_api_url = "https://data.police.uk/api/stops-street"

""" Set the headers """
# This appears to be optional / the default response type
headers = {
    "Accept": "application/json"
}

""" Define query parameters, if any (e.g., search criteria) """
params = {
    "date": "2024-01",
    "poly": "51.536508,-0.021129:51.507804,0.005502:51.508336,0.004562:51.505108,-0.005711:51.487091,-0.008352:51.487045,-0.010779:51.489810,-0.024926:51.502368,-0.029234:51.509395,-0.045493:51.502902,-0.061207:51.506404,-0.073998:51.505460,-0.074663:51.521512,-0.078529:51.535532,-0.062434:51.543330,-0.016550"
}

""" Make the GET request to the FSA API """
r_outcomes = requests.get(outcomes_api_url, headers=headers, params=params)
r_stops = requests.get(stops_api_url, headers=headers, params=params)


""" Parse the response json data and return a Python dictionary """
outcomes = r_outcomes.json()
stops = r_stops.json()

In [27]:
import pandas as pd
import folium
import matplotlib.pyplot as plt
import numpy as np

Store in a pandas Data Frame to clean

In [28]:
stops = pd.DataFrame.from_dict(stops)

Remove null/empty values

In [29]:
stops = stops[stops['age_range'].notnull()]

In [30]:
stops.head()

,age_range,outcome,involved_person,self_defined_ethnicity,gender,legislation,outcome_linked_to_object_of_search,datetime,removal_of_more_than_outer_clothing,outcome_object,location,operation,officer_defined_ethnicity,type,operation_name,object_of_search
0,25-34,A no further action disposal,True,Other ethnic group - Not stated,Male,Misuse of Drugs Act 1971 (section 23),False,2024-01-01T22:41:39+00:00,False,"{'id': 'bu-no-further-action', 'name': 'A no f...","{'latitude': '51.508588', 'street': {'id': 168...",None,Other,Person and Vehicle search,None,Controlled drugs
1,25-34,A no further action disposal,True,White - English/Welsh/Scottish/Northern Irish/...,Male,Misuse of Drugs Act 1971 (section 23),False,2024-01-03T07:40:58+00:00,False,"{'id': 'bu-no-further-action', 'name': 'A no f...","{'latitude': '51.509785', 'street': {'id': 168...",None,White,Person and Vehicle search,None,Controlled drugs
2,over 34,A no further action disposal,True,Other ethnic group - Any other ethnic group,Male,Police and Criminal Evidence Act 1984 (section 1),False,2024-01-03T13:49:28+00:00,False,"{'id': 'bu-no-further-action', 'name': 'A no f...","{'latitude': '51.513647', 'street': {'id': 149...",None,White,Person search,None,Article for use in theft
3,over 34,Arrest,True,White - Any other White background,Male,Police and Criminal Evidence Act 1984 (section 1),True,2024-01-03T18:20:07+00:00,False,"{'id': 'bu-arrest', 'name': 'Arrest'}","{'latitude': '51.513647', 'street': {'id': 149...",None,White,Person search,None,Article for use in theft
4,25-34,Arrest,True,White - English/Welsh/Scottish/Northern Irish/...,Male,Police and Criminal Evidence Act 1984 (section 1),True,2024-01-04T09:09:16+00:00,False,"{'id': 'bu-arrest', 'name': 'Arrest'}","{'latitude': '51.513647', 'street': {'id': 149...",None,White,Person search,None,Article for use in theft


Remove all unnecessary data

In [31]:
stops.columns

Index(['age_range', 'outcome', 'involved_person', 'self_defined_ethnicity',
       'gender', 'legislation', 'outcome_linked_to_object_of_search',
       'datetime', 'removal_of_more_than_outer_clothing', 'outcome_object',
       'location', 'operation', 'officer_defined_ethnicity', 'type',
       'operation_name', 'object_of_search'],
      dtype='object')

In [32]:
stops = stops[['age_range', 'outcome', 'location', 'object_of_search']]

In [33]:
stops

,age_range,outcome,location,object_of_search
0,25-34,A no further action disposal,"{'latitude': '51.508588', 'street': {'id': 168...",Controlled drugs
1,25-34,A no further action disposal,"{'latitude': '51.509785', 'street': {'id': 168...",Controlled drugs
2,over 34,A no further action disposal,"{'latitude': '51.513647', 'street': {'id': 149...",Article for use in theft
3,over 34,Arrest,"{'latitude': '51.513647', 'street': {'id': 149...",Article for use in theft
4,25-34,Arrest,"{'latitude': '51.513647', 'street': {'id': 149...",Article for use in theft
...,...,...,...,...
454,18-24,Arrest,"{'latitude': '51.525424', 'street': {'id': 169...",Stolen goods
455,18-24,A no further action disposal,"{'latitude': '51.515816', 'street': {'id': 168...",Offensive weapons
456,over 34,A no further action disposal,"{'latitude': '51.525582', 'street': {'id': 168...",Controlled drugs
457,25-34,Arrest,"{'latitude': '51.513763', 'street': {'id': 169...",Offensive weapons


Normalise the dictionaries in the column 'location'

In [34]:
stops['latitude'] = stops['location'].str['latitude']
stops['longitude'] = stops['location'].str['longitude']
stops['street_id'] = stops['location'].str['street'].str['id']

In [35]:
stops = stops.drop(['location'], axis=1)

In [36]:
stops

,age_range,outcome,object_of_search,latitude,longitude,street_id
0,25-34,A no further action disposal,Controlled drugs,51.508588,-0.074039,1686590
1,25-34,A no further action disposal,Controlled drugs,51.509785,-0.068095,1687860
2,over 34,A no further action disposal,Article for use in theft,51.513647,-0.074330,1490955
3,over 34,Arrest,Article for use in theft,51.513647,-0.074330,1490955
4,25-34,Arrest,Article for use in theft,51.513647,-0.074330,1490955
...,...,...,...,...,...,...
454,18-24,Arrest,Stolen goods,51.525424,-0.023867,1693544
455,18-24,A no further action disposal,Offensive weapons,51.515816,-0.056482,1689527
456,over 34,A no further action disposal,Controlled drugs,51.525582,-0.059209,1689219
457,25-34,Arrest,Offensive weapons,51.513763,-0.039015,1691842


Save the dataframe as a csv for further investigation

In [37]:
stops.to_csv('stop_and_search.csv')

<hr>

Now for the Outcomes data

In [38]:
outcomes = pd.DataFrame.from_dict(outcomes)

In [39]:
outcomes.head()

,category,date,person_id,crime
0,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'bicycle-theft', 'location_type':..."
1,"{'code': 'court-result-unavailable', 'name': '...",2024-07,None,"{'category': 'drugs', 'location_type': 'Force'..."
2,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'violent-crime', 'location_type':..."
3,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'shoplifting', 'location_type': '..."
4,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'other-theft', 'location_type': '..."


Normalise the data

In [40]:
outcomes = outcomes.join(
    pd.json_normalize(outcomes['category'])
)

In [41]:
outcomes = outcomes.join(pd.json_normalize(outcomes['crime']), lsuffix='_crime')

In [42]:
outcomes.head()

,category_crime,date,person_id,crime,code,name,category,location_type,context,persistent_id,id,location_subtype,month,location.latitude,location.street.id,location.street.name,location.longitude
0,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'bicycle-theft', 'location_type':...",no-further-action,Investigation complete; no suspect identified,bicycle-theft,Force,,790be8053bf55f65ee926104f26daa63f3a4c47bd80216...,115290248,ROAD,2023-12,51.534795,1692295,On or near Old Ford Road,-0.034636
1,"{'code': 'court-result-unavailable', 'name': '...",2024-07,None,"{'category': 'drugs', 'location_type': 'Force'...",court-result-unavailable,Court result unavailable,drugs,Force,,b7029bb109e8099c32dc21df7b903f8bfd06014d05387a...,116091414,ROAD,2024-01,51.526321,1688697,On or near Hague Street,-0.063099
2,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'violent-crime', 'location_type':...",no-further-action,Investigation complete; no suspect identified,violent-crime,Force,,2ce3592324d1563da3a93071ee5b5eecc9a4be02f6afbe...,115275432,ROAD,2023-12,51.525476,1693114,On or near Wellington Way,-0.024845
3,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'shoplifting', 'location_type': '...",no-further-action,Investigation complete; no suspect identified,shoplifting,Force,,cf741cbcdd1c295a5fe176dff10ddf26737ef08dd3c6bb...,116096188,MARKETS,2024-01,51.519193,1686584,On or near Shopping Area,-0.075603
4,"{'code': 'no-further-action', 'name': 'Investi...",2024-01,None,"{'category': 'other-theft', 'location_type': '...",no-further-action,Investigation complete; no suspect identified,other-theft,Force,,d24f1d6b8d2a3f7adfdce5ec23ecc8226f9634e8294502...,115279385,POLICE STATIONS,2023-12,51.518258,1687256,On or near Police Station,-0.071498


Choose the required columns

In [43]:
outcomes = outcomes[['code','name','category', 'location.latitude','location.longitude','location.street.id']]

Save the file as a csv to be used later

In [44]:
outcomes.to_csv('crimes_and_outcomes.csv')